# 02: Feature Extraction and ML Classification

Building on 01_csi_data_explorer.ipynb, this notebook demonstrates the complete feature-based ML pipeline for Wi-Fi CSI motion detection.

## Overview

ESPectre now uses **9 production features** extracted from a turbulence buffer to classify motion in real time with a lightweight neural network (9→32→16→1 MLP).

**Features breakdown:**
- **6 turbulence statistics**: `turb_mean`, `turb_std`, `turb_max`, `turb_min`, `turb_iqr`, `turb_skewness`
- **2 robustness/structure features**: `turb_autocorr`, `turb_mad`
- **1 temporal variation feature**: `waveform_length`

**Why this approach:**
- Turbulence captures **temporal dynamics** of CSI changes (sensitive to motion)
- The production feature set stays **computationally efficient** (MicroPython compatible)
- The final 9-feature subset was selected by long-recording holdout performance, not CV alone
- This notebook now imports the production feature extractor directly to stay aligned with runtime behavior

## 1. Setup & Imports

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import math
from pathlib import Path
from collections import deque

# Setup paths to data and source (relative to notebook location)
NOTEBOOK_DIR = Path('.')
DATA_DIR = NOTEBOOK_DIR / '..' / 'data'
SRC_DIR = NOTEBOOK_DIR / '..' / 'src'

print(f"Notebook directory: ./notebooks/")
print(f"Data directory: ../data/")
print(f"Source directory: ../src/")
print()
print("✓ Setup complete. Ready to load datasets.")
import src.config as config


## 2. Data Loading & Helpers

In [ ]:
# Load dataset metadata
dataset_info_path = DATA_DIR / 'dataset_info.json'
with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

print(f"Dataset version: {dataset_info['format_version']}")
print(f"Updated: {dataset_info['updated_at']}")
print(f"\nAvailable labels: {list(dataset_info['labels'].keys())}")
for label, desc in dataset_info['labels'].items():
    print(f"  {label}: {desc['description']}")

In [ ]:
def load_npz_file(filepath):
    """
    Load CSI data from .npz file.
    
    Returns:
        csi_data: Array of shape (num_packets, 128) — CSI I/Q values
        metadata: Dict with file metadata
    """
    npz = np.load(filepath)
    
    # Get CSI data (main array)
    csi_data = npz['csi_data']
    
    # Extract metadata (handle missing keys gracefully)
    metadata = {
        'num_subcarriers': int(npz.get('num_subcarriers', 64)),
        'num_packets': csi_data.shape[0],
        'duration_ms': float(npz.get('duration_ms', 0)),
        'label': str(npz.get('label', 'unknown')),
        'chip': str(npz.get('chip', 'unknown')),
        'gain_locked': bool(npz.get('gain_locked', False)),
        'collected_at': str(npz.get('collected_at', 'unknown'))
    }
    
    return csi_data, metadata

## 3. Load Data Pair

In [ ]:
# Find a matched baseline/movement pair from the dataset
# We'll use the first pair from dataset_info that has optimal_pair_movement_file set

baseline_files = dataset_info['files']['baseline']
movement_files_by_name = {f['filename']: f for f in dataset_info['files']['movement']}

# Find a pair
baseline_entry = None
movement_entry = None

for entry in baseline_files:
    if 'optimal_pair_movement_file' in entry:
        pair_name = entry['optimal_pair_movement_file']
        if pair_name in movement_files_by_name:
            baseline_entry = entry
            movement_entry = movement_files_by_name[pair_name]
            break

if baseline_entry is None:
    # Fallback: use any baseline/movement pair
    baseline_entry = baseline_files[0]
    movement_entry = dataset_info['files']['movement'][0]

baseline_file = baseline_entry['filename']
movement_file = movement_entry['filename']
chip = baseline_entry['chip']

print(f"Loading pair:")
print(f"  Baseline: {baseline_file}")
print(f"  Movement: {movement_file}")
print(f"  Chip: {chip}")

In [ ]:
# Load the data
baseline_path = DATA_DIR / 'baseline' / baseline_file
movement_path = DATA_DIR / 'movement' / movement_file

baseline_csi, baseline_meta = load_npz_file(baseline_path)
movement_csi, movement_meta = load_npz_file(movement_path)

print(f"Baseline: {baseline_meta['num_packets']} packets, {baseline_meta['duration_ms']:.1f} ms")
print(f"  Chip: {baseline_meta['chip']}, Gain locked: {baseline_meta['gain_locked']}")
print(f"\nMovement: {movement_meta['num_packets']} packets, {movement_meta['duration_ms']:.1f} ms")
print(f"  Chip: {movement_meta['chip']}, Gain locked: {movement_meta['gain_locked']}")

# Combine for analysis (vertically stack CSI arrays)
all_csi = np.vstack([baseline_csi, movement_csi])
labels = np.concatenate([
    np.zeros(baseline_meta['num_packets']),  # 0 = baseline
    np.ones(movement_meta['num_packets'])    # 1 = movement
])

print(f"\nCombined: {all_csi.shape[0]} packets, {all_csi.shape[1]} I/Q values per packet")

## 4. Turbulence Buffer Construction

In [ ]:
# ML-optimized subcarrier set (fixed across all models)
# These were selected via grid search to maximize detection accuracy
DEFAULT_SUBCARRIERS = config.DEFAULT_SUBCARRIERS

print(f"Using {len(DEFAULT_SUBCARRIERS)} subcarriers for ML: {DEFAULT_SUBCARRIERS}")

def extract_amplitude(csi_packet, subcarrier_indices=None):
    """Extract amplitude from I/Q pairs for selected subcarriers."""
    # CSI packet has shape (128,) = 64 subcarriers × 2 (Q, I)
    csi_reshaped = csi_packet.reshape(64, 2)
    Q = csi_reshaped[:, 0].astype(np.float32)
    I = csi_reshaped[:, 1].astype(np.float32)

    amplitude = np.sqrt(I**2 + Q**2)

    if subcarrier_indices is not None:
        amplitude = amplitude[subcarrier_indices]

    return amplitude

def compute_spatial_turbulence(amplitudes, use_cv_normalization=True):
    """
    Compute spatial turbulence with optional CV normalization.

    Args:
        amplitudes: array of amplitude values
        use_cv_normalization: True = std/mean (CV, gain-invariant)
                             False = raw std (gain-locked chips)

    SOURCE: src/segmentation.py:compute_spatial_turbulence()
    """
    if len(amplitudes) < 2:
        return 0.0

    mean = np.mean(amplitudes)
    if mean < 1e-10:
        return 0.0

    std = np.std(amplitudes)

    if use_cv_normalization:
        # CV normalization: std/mean (gain-invariant)
        return std / mean
    else:
        # Raw std: better sensitivity for gain-locked chips
        return std

# Determine turbulence mode based on gain_locked metadata
# SOURCE: Real code defaults use_cv_normalization based on chip capability
use_cv_baseline = not baseline_meta['gain_locked']
use_cv_movement = not movement_meta['gain_locked']

print(f"\nTurbulence mode selection:")
print(f"  Baseline ({baseline_meta['chip'].upper()}): {'CV (gain-invariant)' if use_cv_baseline else 'Raw Std (gain-locked)'}")
print(f"  Movement ({movement_meta['chip'].upper()}): {'CV (gain-invariant)' if use_cv_movement else 'Raw Std (gain-locked)'}")

# Compute turbulence and keep per-packet amplitudes for analysis/plots.
# The current production model uses 9 features, but amplitudes are still useful
# here for exploratory visualization of the CSI signal itself.
turbulence_series = []
amplitude_series = []
for packet_idx in range(all_csi.shape[0]):
    csi_packet = all_csi[packet_idx]
    amps = extract_amplitude(csi_packet, DEFAULT_SUBCARRIERS)
    amplitude_series.append(amps)
    # Use appropriate mode based on source dataset
    use_cv = use_cv_baseline if packet_idx < baseline_meta['num_packets'] else use_cv_movement
    turb = compute_spatial_turbulence(amps, use_cv_normalization=use_cv)
    turbulence_series.append(turb)

turbulence_series = np.array(turbulence_series)
amplitude_series = np.array(amplitude_series)  # shape: (num_packets, 12)

baseline_end_idx = baseline_meta['num_packets']

print(f"\nTurbulence series shape: {turbulence_series.shape}")
print(f"Amplitude series shape: {amplitude_series.shape}")
print(f"\nBaseline turbulence:")
baseline_turb = turbulence_series[:baseline_end_idx]
print(f"  Mean: {baseline_turb.mean():.4f}, Std: {baseline_turb.std():.4f}")
print(f"\nMovement turbulence:")
movement_turb = turbulence_series[baseline_end_idx:]
print(f"  Mean: {movement_turb.mean():.4f}, Std: {movement_turb.std():.4f}")


In [ ]:
# Visualize the turbulence time series
fig, ax = plt.subplots(figsize=(14, 4))

x = np.arange(len(turbulence_series))
baseline_end = baseline_meta['num_packets']

# Plot baseline region
ax.plot(x[:baseline_end], turbulence_series[:baseline_end], 
        color='blue', linewidth=1, alpha=0.7, label='Baseline')

# Plot movement region
ax.plot(x[baseline_end:], turbulence_series[baseline_end:], 
        color='red', linewidth=1, alpha=0.7, label='Movement')

# Add vertical separator
ax.axvline(baseline_end, color='gray', linestyle='--', linewidth=2, alpha=0.5)

# Add background shading for clarity
ax.axvspan(0, baseline_end, alpha=0.05, color='blue')
ax.axvspan(baseline_end, len(turbulence_series), alpha=0.05, color='red')

ax.set_xlabel('Packet Index', fontsize=12)
ax.set_ylabel('Turbulence (std/mean)', fontsize=12)
ax.set_title('Turbulence Time Series: Baseline vs Movement', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

print(f"✓ Turbulence clearly separates baseline (blue, lower) from movement (red, higher)")

## 5. Feature Extraction Pipeline

This notebook now uses the current 9-feature production extractor directly from `src/features.py`, so the tutorial stays aligned with the shipped ML model.

In [ ]:
# Use the production feature extractor directly so the notebook stays aligned.
from src.features import DEFAULT_FEATURES, extract_features_by_name

WINDOW_SIZE = 100  # packets per feature window
WINDOW_STRIDE = 1  # slide window by 1 packet (heavy overlap for smooth features)
FEATURE_NAMES = list(DEFAULT_FEATURES)

print("Production feature extractor loaded from src.features")
print(f"Feature set ({len(FEATURE_NAMES)}): {', '.join(FEATURE_NAMES)}")


# ----- Extract features for all sliding windows -----
print(f"\nExtracting features: window={WINDOW_SIZE}, stride={WINDOW_STRIDE}")

feature_matrix = []
window_labels = []

total_packets = len(turbulence_series)

for start in range(0, total_packets - WINDOW_SIZE + 1, WINDOW_STRIDE):
    end = start + WINDOW_SIZE
    window_turb = turbulence_series[start:end]

    # Extract the current production feature vector for this window.
    features = extract_features_by_name(
        window_turb,
        len(window_turb),
        feature_names=FEATURE_NAMES,
    )
    feature_matrix.append(features)

    # Label: use center of window to determine baseline vs movement
    center = start + WINDOW_SIZE // 2
    label = 0 if center < baseline_end_idx else 1  # 0=baseline, 1=movement
    window_labels.append(label)

feature_matrix = np.array(feature_matrix)
window_labels = np.array(window_labels)

baseline_count = np.sum(window_labels == 0)
movement_count = np.sum(window_labels == 1)

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"  Baseline windows: {baseline_count}")
print(f"  Movement windows: {movement_count}")
print(f"  Total windows: {len(window_labels)}")

In [ ]:
# Display feature statistics
baseline_features = feature_matrix[window_labels == 0]
movement_features = feature_matrix[window_labels == 1]

print("Feature Statistics (Baseline vs Movement):")
print("\n" + "="*90)
print(f"{'Feature':<20} {'Baseline Mean':<18} {'Movement Mean':<18} {'Difference':<15}")
print("="*90)

for i, name in enumerate(FEATURE_NAMES):
    baseline_mean = np.mean(baseline_features[:, i])
    movement_mean = np.mean(movement_features[:, i])
    diff = movement_mean - baseline_mean
    print(f"{name:<20} {baseline_mean:<18.6f} {movement_mean:<18.6f} {diff:+.6f}")

print("="*90)

## 6. Feature Visualization

In [ ]:
# Plot each feature over time using a dynamic grid.
n_features = len(FEATURE_NAMES)
n_cols = 3
n_rows = int(np.ceil(n_features / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

baseline_count = np.sum(window_labels == 0)

for feature_idx, feature_name in enumerate(FEATURE_NAMES):
    ax = axes[feature_idx]

    x = np.arange(len(window_labels))

    # Plot baseline region
    ax.plot(x[:baseline_count], feature_matrix[:baseline_count, feature_idx],
            color='blue', linewidth=1, alpha=0.7, marker='o', markersize=3)

    # Plot movement region
    ax.plot(x[baseline_count:], feature_matrix[baseline_count:, feature_idx],
            color='red', linewidth=1, alpha=0.7, marker='o', markersize=3)

    # Add vertical separator
    ax.axvline(baseline_count, color='gray', linestyle='--', linewidth=1, alpha=0.5)

    # Annotate means
    baseline_mean = np.mean(feature_matrix[:baseline_count, feature_idx])
    movement_mean = np.mean(feature_matrix[baseline_count:, feature_idx])
    ax.axhline(baseline_mean, color='blue', linestyle=':', alpha=0.5, linewidth=1)
    ax.axhline(movement_mean, color='red', linestyle=':', alpha=0.5, linewidth=1)

    ax.set_title(feature_name, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.set_xlabel('Window Index', fontsize=9)
    ax.set_ylabel('Value', fontsize=9)

for ax in axes[n_features:]:
    ax.axis('off')

plt.tight_layout()
plt.suptitle('Feature Evolution Over Time: Baseline vs Movement',
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("Blue = baseline, Red = movement. Dashed lines = window means.")

## 7. Feature Distributions & Discriminative Power

In [ ]:
def cohens_d(group1, group2):
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group2) - np.mean(group1)) / pooled_std if pooled_std > 0 else 0.0


# Calculate effect sizes
effect_sizes = []
for i in range(len(FEATURE_NAMES)):
    d = cohens_d(baseline_features[:, i], movement_features[:, i])
    effect_sizes.append(abs(d))

# Sort by effect size
sorted_indices = np.argsort(effect_sizes)[::-1]

print("\nFeature Discriminative Power (Cohen's d, sorted):")
print("\n" + "="*70)
print(f"{'Rank':<6} {'Feature':<20} {'Effect Size':<15} {'Interpretation':<25}")
print("="*70)

interpretations = {
    lambda x: x < 0.2: 'Negligible',
    lambda x: x < 0.5: 'Small',
    lambda x: x < 0.8: 'Medium',
    lambda x: True: 'Large'
}

for rank, idx in enumerate(sorted_indices, 1):
    feature_name = FEATURE_NAMES[idx]
    d = effect_sizes[idx]
    interpretation = next(v for k, v in interpretations.items() if k(d))
    print(f"{rank:<6} {feature_name:<20} {d:<15.4f} {interpretation:<25}")

print("="*70)

In [ ]:
# Box/violin plots for top 6 features by effect size
top_6_indices = sorted_indices[:6]
top_6_names = [FEATURE_NAMES[i] for i in top_6_indices]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for plot_idx, feature_idx in enumerate(top_6_indices):
    ax = axes[plot_idx]
    feature_name = FEATURE_NAMES[feature_idx]
    
    data_to_plot = [
        baseline_features[:, feature_idx],
        movement_features[:, feature_idx]
    ]
    
    parts = ax.violinplot(data_to_plot, positions=[0, 1], showmeans=True)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Baseline', 'Movement'])
    ax.set_title(f"{feature_name} (d={effect_sizes[feature_idx]:.3f})", 
                 fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylabel('Value', fontsize=10)

plt.tight_layout()
plt.suptitle('Top 6 Features by Discriminative Power (Cohen\'s d)', 
             fontsize=13, fontweight='bold', y=1.00)
plt.show()

## 8. Feature Correlation Matrix

In [ ]:
# Compute correlation matrix
correlation_matrix = np.corrcoef(feature_matrix.T)

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')

# Set ticks and labels
ax.set_xticks(np.arange(len(FEATURE_NAMES)))
ax.set_yticks(np.arange(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right')
ax.set_yticklabels(FEATURE_NAMES)

# Add correlation values to cells
for i in range(len(FEATURE_NAMES)):
    for j in range(len(FEATURE_NAMES)):
        val = correlation_matrix[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=8)

plt.colorbar(im, ax=ax, label='Correlation Coefficient')
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Print high correlations
print("\nHighly Correlated Feature Pairs (|r| > 0.7):")
print("="*60)
for i in range(len(FEATURE_NAMES)):
    for j in range(i+1, len(FEATURE_NAMES)):
        corr = correlation_matrix[i, j]
        if abs(corr) > 0.7:
            print(f"{FEATURE_NAMES[i]:<20} <-> {FEATURE_NAMES[j]:<20} : {corr:+.3f}")
print("="*60)

## 9. ML Classification: Simple Threshold Baseline

In [ ]:
# Simple baseline: threshold on turb_std
turb_std_idx = FEATURE_NAMES.index('turb_std')
threshold = np.percentile(feature_matrix[window_labels == 0, turb_std_idx], 95)

print(f"Threshold-based classifier (turb_std > {threshold:.4f}):")
print()

predictions_threshold = (feature_matrix[:, turb_std_idx] > threshold).astype(int)
# window_labels is already 0=baseline, 1=movement from the extraction loop
true_labels_binary = window_labels.astype(int)

tp = np.sum((predictions_threshold == 1) & (true_labels_binary == 1))
tn = np.sum((predictions_threshold == 0) & (true_labels_binary == 0))
fp = np.sum((predictions_threshold == 1) & (true_labels_binary == 0))
fn = np.sum((predictions_threshold == 0) & (true_labels_binary == 1))

accuracy_thr = (tp + tn) / (tp + tn + fp + fn)
precision_thr = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_thr = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_thr = 2 * (precision_thr * recall_thr) / (precision_thr + recall_thr) if (precision_thr + recall_thr) > 0 else 0

print(f"Accuracy:  {accuracy_thr:.4f}")
print(f"Precision: {precision_thr:.4f}")
print(f"Recall:    {recall_thr:.4f}")
print(f"F1 Score:  {f1_thr:.4f}")
print()
print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

## 10. ML Classification: Neural Network Inference

In [ ]:
# Load ML weights from the generated export used by runtime and tests.
import sys
from pathlib import Path

notebook_dir = Path.cwd()
for src_dir in (notebook_dir / "src", notebook_dir.parent / "src"):
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

try:
    from src.ml_weights import (
        FEATURE_MEAN,
        FEATURE_SCALE,
        WEIGHTS,
        BIASES,
        MODEL_LAYER_SIZES,
        MODEL_HIDDEN_LAYERS,
        NORMALIZATION_MODE,
    )
except ImportError:
    from ml_weights import (
        FEATURE_MEAN,
        FEATURE_SCALE,
        WEIGHTS,
        BIASES,
        MODEL_LAYER_SIZES,
        MODEL_HIDDEN_LAYERS,
        NORMALIZATION_MODE,
    )

FEATURE_MEAN = np.asarray(FEATURE_MEAN, dtype=np.float32)
FEATURE_SCALE = np.asarray(FEATURE_SCALE, dtype=np.float32)
WEIGHTS = [np.asarray(weights, dtype=np.float32) for weights in WEIGHTS]
BIASES = [np.asarray(biases, dtype=np.float32) for biases in BIASES]

print("ML model weights loaded successfully")
print(f"Architecture: {' -> '.join(str(x) for x in MODEL_LAYER_SIZES)}")
print(f"Hidden layers: {MODEL_HIDDEN_LAYERS}")
print(f"Normalization: {NORMALIZATION_MODE}")
for idx, (weights, biases) in enumerate(zip(WEIGHTS, BIASES), start=1):
    print(f"Layer {idx}: W{idx}={weights.shape}, B{idx}={biases.shape}")

In [ ]:
def sigmoid(x):
    """Sigmoid activation function."""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def relu(x):
    """ReLU activation function."""
    return np.maximum(0, x)


def mlp_predict(features, weights, biases, feature_mean, feature_scale):
    """
    Generic exported-model forward pass for motion detection.

    Mirrors the generated runtime structure in `src/ml_detector.py`,
    but keeps notebook probabilities on [0, 1] for readability.
    """
    activations = (features - feature_mean) / feature_scale

    for layer_idx, (layer_weights, layer_biases) in enumerate(zip(weights, biases)):
        activations = activations @ layer_weights + layer_biases
        if layer_idx < len(weights) - 1:
            activations = relu(activations)

    logits = activations.reshape(-1)
    probabilities = sigmoid(logits)
    return logits, probabilities


In [ ]:
# Run MLP inference on all feature windows using the exported model layout
logits, probabilities = mlp_predict(
    feature_matrix,
    WEIGHTS,
    BIASES,
    FEATURE_MEAN,
    FEATURE_SCALE,
)

# Binary predictions at threshold 0.5
predictions_ml = (probabilities >= 0.5).astype(int)

print(f"MLP inference complete on {len(probabilities)} windows")
print(f"  Predicted movement: {np.sum(predictions_ml == 1)} windows")
print(f"  Predicted baseline: {np.sum(predictions_ml == 0)} windows")
print(f"  Probability range: [{probabilities.min():.4f}, {probabilities.max():.4f}]")

In [ ]:
# Compute classification metrics
tp = np.sum((predictions_ml == 1) & (true_labels_binary == 1))
tn = np.sum((predictions_ml == 0) & (true_labels_binary == 0))
fp = np.sum((predictions_ml == 1) & (true_labels_binary == 0))
fn = np.sum((predictions_ml == 0) & (true_labels_binary == 1))

accuracy_ml = (tp + tn) / (tp + tn + fp + fn)
precision_ml = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_ml = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_ml = 2 * (precision_ml * recall_ml) / (precision_ml + recall_ml) if (precision_ml + recall_ml) > 0 else 0

print("\n" + "="*70)
print("ML MODEL PERFORMANCE (9->24->12->1 MLP)")
print("="*70)
print(f"Accuracy:  {accuracy_ml:.4f} (97.5% or higher is typical)")
print(f"Precision: {precision_ml:.4f} (minimize false alarms)")
print(f"Recall:    {recall_ml:.4f} (catch all motion events)")
print(f"F1 Score:  {f1_ml:.4f}")
print()
print(f"True Positives:  {tp}")
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print("="*70)

In [ ]:
# Plot ML probabilities over time
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(window_labels))

# Plot baseline region
ax.plot(x[:baseline_count], probabilities[:baseline_count],
        color='blue', linewidth=1.5, alpha=0.7, label='Baseline')

# Plot movement region
ax.plot(x[baseline_count:], probabilities[baseline_count:],
        color='red', linewidth=1.5, alpha=0.7, label='Movement')

# Add decision threshold line
ax.axhline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')

# Add vertical separator
ax.axvline(baseline_count, color='gray', linestyle='--', linewidth=2, alpha=0.5)

# Shade regions
ax.axhspan(0, 0.5, alpha=0.1, color='blue', label='Baseline Region')
ax.axhspan(0.5, 1, alpha=0.1, color='red', label='Motion Region')

ax.set_xlabel('Window Index', fontsize=12)
ax.set_ylabel('Motion Probability', fontsize=12)
ax.set_title('MLP Motion Probabilities Over Time', fontsize=14, fontweight='bold')
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

print(f"Blue region: low motion probability (baseline)")
print(f"Red region: high motion probability (movement)")
print(f"The MLP successfully separates the two states with clear margin.")

## 11. Confusion Matrix & Error Analysis

In [ ]:
# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))

# Confusion matrix
cm = np.array([[tn, fp], [fn, tp]])

im = ax.imshow(cm, cmap='Blues', aspect='auto')

# Labels and ticks
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Baseline', 'Predicted Motion'])
ax.set_yticklabels(['Actual Baseline', 'Actual Motion'])

# Values in cells
for i in range(2):
    for j in range(2):
        text = ax.text(j, i, f'{cm[i, j]}', ha='center', va='center',
                        color='white' if cm[i, j] > cm.max() / 2 else 'black',
                        fontsize=16, fontweight='bold')

plt.colorbar(im, ax=ax)
ax.set_title('ML Classifier Confusion Matrix', fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_xlabel('Prediction', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze errors
false_positives = np.where((predictions_ml == 1) & (true_labels_binary == 0))[0]
false_negatives = np.where((predictions_ml == 0) & (true_labels_binary == 1))[0]

print("\nERROR ANALYSIS")
print("="*70)

if len(false_positives) > 0:
    print(f"\nFalse Positives ({len(false_positives)} windows):")
    print(f"  Window indices: {false_positives[:10]}{'...' if len(false_positives) > 10 else ''}")
    fp_probs = probabilities[false_positives]
    print(f"  Avg probability: {np.mean(fp_probs):.4f}")
    print(f"  These are baseline windows classified as motion (conservative)")
else:
    print(f"\nNo false positives! Perfect baseline detection.")

if len(false_negatives) > 0:
    print(f"\nFalse Negatives ({len(false_negatives)} windows):")
    print(f"  Window indices: {false_negatives[:10]}{'...' if len(false_negatives) > 10 else ''}")
    fn_probs = probabilities[false_negatives]
    print(f"  Avg probability: {np.mean(fn_probs):.4f}")
    print(f"  These are motion windows classified as baseline (missed detections)")
else:
    print(f"\nNo false negatives! Perfect motion detection.")

print("="*70)

## 12. ROC-like Analysis: Threshold vs Performance

In [ ]:
# Sweep over thresholds and compute metrics
thresholds = np.linspace(0, 1, 101)
accuracies = []
precisions = []
recalls = []
f1_scores = []

for threshold in thresholds:
    preds = (probabilities >= threshold).astype(int)
    tp = np.sum((preds == 1) & (true_labels_binary == 1))
    tn = np.sum((preds == 0) & (true_labels_binary == 0))
    fp = np.sum((preds == 1) & (true_labels_binary == 0))
    fn = np.sum((preds == 0) & (true_labels_binary == 1))
    
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    
    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Metrics vs Threshold
ax1.plot(thresholds, accuracies, label='Accuracy', linewidth=2)
ax1.plot(thresholds, precisions, label='Precision', linewidth=2)
ax1.plot(thresholds, recalls, label='Recall', linewidth=2)
ax1.plot(thresholds, f1_scores, label='F1 Score', linewidth=2)
ax1.axvline(0.5, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Default (0.5)')
ax1.set_xlabel('Decision Threshold', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Classification Metrics vs Decision Threshold', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)
ax1.set_ylim([0, 1.05])

# Plot 2: Precision vs Recall (PR curve)
ax2.plot(recalls, precisions, linewidth=2, marker='o', markersize=3)
ax2.axvline(recall_ml, color='red', linestyle='--', alpha=0.5, label=f'Threshold=0.5 (Recall={recall_ml:.3f})')
ax2.axhline(precision_ml, color='red', linestyle='--', alpha=0.5, label=f'(Precision={precision_ml:.3f})')
ax2.set_xlabel('Recall (True Positive Rate)', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)
ax2.set_xlim([0, 1.05])
ax2.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print(f"Optimal F1 at threshold={thresholds[np.argmax(f1_scores)]:.2f} (F1={np.max(f1_scores):.4f})")

## 13. Summary & Key Insights

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                    FEATURE EXTRACTION & ML SUMMARY                         ║
╚════════════════════════════════════════════════════════════════════════════╝

1. TURBULENCE BUFFER CONSTRUCTION
   ─────────────────────────────────
   • Turbulence is computed packet-by-packet from the fixed 12 ML subcarriers
   • Computed per packet -> creates temporal sequence
   • Window size: 100 packets -> captures ~100 ms of dynamics
   • Baseline turbulence stays lower/stabler than movement turbulence

2. FEATURE EXTRACTION
   ─────────────────────────────────
   • 9 production features total:
     - 6 statistics: mean, std, max, min, IQR, skewness
     - 2 robustness/structure features: autocorr, MAD
     - 1 temporal variation feature: waveform_length
   • Feature extraction is imported directly from src/features.py

3. FEATURE IMPORTANCE
   ─────────────────────────────────
   • Top discriminative features (by Cohen's d):
""")

# Print top 5 features
for rank, idx in enumerate(sorted_indices[:5], 1):
    print(f"     {rank}. {FEATURE_NAMES[idx]:<20} d = {effect_sizes[idx]:.3f}")

print("""
4. ML CLASSIFICATION PERFORMANCE
   ─────────────────────────────────
   • Architecture: 9 -> 24 (ReLU) -> 12 (ReLU) -> 1 (Sigmoid)
   • Features normalized: (x - mean) / std using training statistics
   • Production training uses grouped CV plus early stopping

   Results on this dataset:
""")

print(f"     • Accuracy:  {accuracy_ml:.4f}")
print(f"     • Precision: {precision_ml:.4f} (low false alarm rate)")
print(f"     • Recall:    {recall_ml:.4f} (high detection rate)")
print(f"     • F1 Score:  {f1_ml:.4f}")
print(f"     • Threshold: 0.50 (adjustable for different sensitivity)")

print("""
5. FEATURE CORRELATION
   ─────────────────────────────────
   • The kept features are still partially correlated (for example IQR, MAD, std)
   • The production set was chosen by long-recording holdout behavior, not by correlation alone
   • The dropped features were turb_kurtosis, turb_entropy, and turb_slope

6. NEXT STEPS
   ─────────────────────────────────
   • Use trained weights in ESPectre firmware (src/ml_weights.h)
   • Test on real-world scenarios (different rooms, distances, materials)
   • Optimize for embedded systems (quantization, pruning possible)
   • Extend to gesture recognition (different motion types)
   • Integrate with CSI collection pipeline for continuous monitoring

7. REFERENCES
   ─────────────────────────────────
   • Features defined in: src/features.py
   • Training code: tools/10_train_ml_model.py
   • ESP-IDF integration: components/espectre/
   • See ALGORITHMS.md for mathematical details

╔════════════════════════════════════════════════════════════════════════════╗
║  The current production MLP uses a compact 9-feature input and still      ║
║  separates idle and motion clearly on this walkthrough dataset.           ║
║  Holdout long-recording validation remains the real promotion gate.       ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Final comparison table
print("\n" + "="*80)
print("CLASSIFIER COMPARISON")
print("="*80)
print(f"{'Method':<25} {'Accuracy':<15} {'Precision':<15} {'Recall':<15} {'F1':<15}")
print("-"*80)
print(f"{'Threshold (turb_std)':<25} {accuracy_thr:<15.4f} {precision_thr:<15.4f} {recall_thr:<15.4f} {f1_thr:<15.4f}")
print(f"{'MLP (9→32→16→1)':<25} {accuracy_ml:<15.4f} {precision_ml:<15.4f} {recall_ml:<15.4f} {f1_ml:<15.4f}")
print("="*80)
print(f"\nConclusion: The neural network significantly outperforms simple thresholding,")
print(f"achieving {f1_ml:.1%} F1 score through learned nonlinear feature combinations.")
